## Project Introduction

This project focuses on analyzing global COVID-19 data using two primary datasets:

1. **Grouped Data (df1)**  
   - Contains time-series information such as confirmed cases, deaths, recoveries, and daily changes.  
   - Includes country-level data along with WHO region classification.  
   - Useful for analyzing trends over time.

2. **Worldometer Data (df2)**  
   - Provides a snapshot of COVID-19 statistics by country.  
   - Includes metrics such as total cases, deaths, recoveries, population, testing data, and per-million indicators.  
   - Useful for comparative and statistical analysis across countries.

### Objective

The goal of this project is to:
- Clean and preprocess both datasets to ensure data quality and consistency
- Perform necessary transformations for analysis
- Build a structured data model
- Create an interactive Power BI dashboard to visualize:
  - Global trends over time
  - Country-wise comparisons
  - Regional insights based on WHO classifications

This project demonstrates the integration of Python (Pandas) for data preprocessing and Power BI for data visualization and reporting.

In [1]:
import pandas as pd 

In [2]:
df1 = pd.read_csv('full_grouped.csv')
df2 = pd.read_csv('worldometer_data.csv')

In [3]:
df1.head()
print(f"This dataset has {df1.shape[0]} rows and {df1.shape[1]} columns")

This dataset has 35156 rows and 10 columns


In [4]:
df2.head()
print(f"This dataset has {df2.shape[0]} rows and {df2.shape[1]} columns")

This dataset has 209 rows and 16 columns


In [5]:
df1.columns

Index(['Date', 'Country/Region', 'Confirmed', 'Deaths', 'Recovered', 'Active',
       'New cases', 'New deaths', 'New recovered', 'WHO Region'],
      dtype='object')

In [6]:
df1.isnull().sum()
df1.duplicated().sum()
df1.dtypes #since date is object here we convert it to datetime
df1['Date'] = pd.to_datetime(df1['Date'])

In [7]:
#check if dataframe has any negative value because we do not want e.g. deaths = -1 which doesn't make sense 
numeric_cols = ['Confirmed','Deaths','Recovered','Active','New cases','New deaths','New recovered']
neg_cols = []
for col in numeric_cols:
    if (df1[col] < 0).any():
        neg_cols.append(col)

if neg_cols:
    print("Negative values found in columns:")
    print(neg_cols)
else:
    print("No negative values found in any column")
#Check negative total values in each column
for col in ['Active', 'New deaths', 'New recovered']:
    print(col, (df1[col] < 0).sum())

Negative values found in columns:
['Active', 'New deaths', 'New recovered']
Active 2
New deaths 38
New recovered 77


since negative values are found now we clean them below

In [8]:
# fill negative values with 0 and final view if any negative values remain 
df1[['Active', 'New deaths', 'New recovered']] = df1[
    ['Active', 'New deaths', 'New recovered']
].clip(lower=0)

for col in ['Active', 'New deaths', 'New recovered']:
    print(col, (df1[col] < 0).sum())

Active 0
New deaths 0
New recovered 0


In [9]:
df1.rename(columns={
    'Country/Region': 'Country_Region',
    'WHO Region': 'WHO_Region',
    'New cases': 'New_Cases',
    'New deaths': 'New_Deaths',
    'New recovered': 'New_Recovered'
}, inplace=True)

**Conclusion for grouped_data (df1):**

The dataset required minimal preprocessing. The `Date` column was converted from object to datetime format to ensure proper time-series analysis. Additionally, negative values and naming issues present in certain columns were identified and handled appropriately to maintain data consistency and reliability. 

In [10]:
df2.isnull().sum()
df2.duplicated().sum()

np.int64(0)

In [20]:
df2['Country_Region'].nunique() == len(df2)

True

In [11]:
df2[['NewCases', 'NewDeaths', 'NewRecovered']] = df2[
    ['NewCases', 'NewDeaths', 'NewRecovered']
].fillna(0)

In [12]:
df2[['TotalDeaths', 'TotalRecovered', 'ActiveCases']] = df2[
    ['TotalDeaths', 'TotalRecovered', 'ActiveCases']
].fillna(0)

In [13]:
df2['Continent'] = df2['Continent'].fillna('Unknown')
df2['WHO Region'] = df2['WHO Region'].fillna('Unknown')

In [14]:
df2['Population'] = df2['Population'].fillna(df2['Population'].median())

In [15]:
cols = ['Tot Cases/1M pop', 'Deaths/1M pop', 'Tests/1M pop']
df2[cols] = df2[cols].fillna(0)

In [16]:
df2['TotalTests'] = df2['TotalTests'].fillna(0)

In [17]:
df2.rename(columns={
    'Country/Region': 'Country_Region',
    'WHO Region': 'WHO_Region',
    'Tot Cases/1M pop': 'Cases_per_1M',
    'Deaths/1M pop': 'Deaths_per_1M',
    'Tests/1M pop': 'Tests_per_1M',

    'TotalCases': 'Total_Cases',
    'NewCases': 'New_Cases',
    'TotalDeaths': 'Total_Deaths',
    'NewDeaths': 'New_Deaths',
    'TotalRecovered': 'Total_Recovered',
    'NewRecovered': 'New_Recovered',
    'ActiveCases': 'Active_Cases',
    'TotalTests': 'Total_Tests'
}, inplace=True)

In [18]:
df2['Serious_Critical'] = df2['Serious,Critical'].fillna(0)
df2.drop(columns=['Serious,Critical'])

,Country_Region,Continent,Population,Total_Cases,New_Cases,Total_Deaths,New_Deaths,Total_Recovered,New_Recovered,Active_Cases,Cases_per_1M,Deaths_per_1M,Total_Tests,Tests_per_1M,WHO_Region,Serious_Critical
0,USA,North America,3.311981e+08,5032179,0.0,162804.0,0.0,2576668.0,0.0,2292707.0,15194.0,492.0,63139605.0,190640.0,Americas,18296.0
1,Brazil,South America,2.127107e+08,2917562,0.0,98644.0,0.0,2047660.0,0.0,771258.0,13716.0,464.0,13206188.0,62085.0,Americas,8318.0
2,India,Asia,1.381345e+09,2025409,0.0,41638.0,0.0,1377384.0,0.0,606387.0,1466.0,30.0,22149351.0,16035.0,South-EastAsia,8944.0
3,Russia,Europe,1.459409e+08,871894,0.0,14606.0,0.0,676357.0,0.0,180931.0,5974.0,100.0,29716907.0,203623.0,Europe,2300.0
4,South Africa,Africa,5.938157e+07,538184,0.0,9604.0,0.0,387316.0,0.0,141264.0,9063.0,162.0,3149807.0,53044.0,Africa,539.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
204,Montserrat,North America,4.992000e+03,13,0.0,1.0,0.0,10.0,0.0,2.0,2604.0,200.0,61.0,12220.0,Unknown,0.0
205,Caribbean Netherlands,North America,2.624700e+04,13,0.0,0.0,0.0,7.0,0.0,6.0,495.0,0.0,424.0,16154.0,Unknown,0.0
206,Falkland Islands,South America,3.489000e+03,13,0.0,0.0,0.0,13.0,0.0,0.0,3726.0,0.0,1816.0,520493.0,Unknown,0.0
207,Vatican City,Europe,8.010000e+02,12,0.0,0.0,0.0,12.0,0.0,0.0,14981.0,0.0,0.0,0.0,Europe,0.0


### Column Transformation

- The column `Serious,Critical` contained inconsistent naming and formatting issues.
- It was cleaned and renamed to `Serious_Critical` for better usability and compatibility with Power BI.

- After successful transformation, the original column was dropped to:
  - Avoid redundancy
  - Maintain a clean dataset schema

**Conclusion for worldometer_data(df2)**: requires data cleaning due to:
  - Inconsistent data formats (e.g., '+' symbols in numeric columns)
  - Incorrect data types
  - Column naming issues

In [19]:
#Exporting both cleaned datasets as CSV
#grouped_data(df1)
try:
    df1.to_csv("grouped_data_cleaned.csv", index=False)
    print("grouped_data exported successfully!")

except Exception as e:
    print("Error exporting grouped_data:", e)

#worldometer_data(df2)
try:
    df2.to_csv("worldometer_data_cleaned.csv", index=False)
    print("worldometer_data exported successfully!")

except Exception as e:
    print("Error exporting worldometer_data:", e)

grouped_data exported successfully!
worldometer_data exported successfully!


## Final Conclusion

- The grouped dataset (df1) was validated and required minimal preprocessing.  
  - No duplicate records were found.  
  - Data quality was consistent, with only minor validation checks performed (e.g., data types and negative values and Non-standard column naming conventions).

- The worldometer dataset (df2) required significant data cleaning due to:
  - Missing values across multiple columns
  - Inconsistent formatting in numeric fields (e.g., '+' symbols)
  - Non-standard column naming conventions

- Key preprocessing steps included:
  - Handling missing values using appropriate strategies (zero imputation, median imputation, and categorical labeling as 'Unknown')
  - Converting data types for accurate numerical analysis
  - Renaming columns to ensure consistency and compatibility
  - Transforming and standardizing problematic fields (e.g., `Serious,Critical` to `Serious_Critical`)

- Negative values in daily metrics were treated as data inconsistencies and adjusted to ensure meaningful analysis and visualization.

- Final cleaned datasets were prepared for integration into Power BI, where:
  - Data modeling was performed
  - Relationships were established
  - Measures and visualizations were created to support analytical insights

This project highlights a complete data analysis workflow, from raw data preprocessing to building an interactive business intelligence dashboard.